## **2. Preprocesamiento**

**Objetivo:** transformar los datos crudos (texto y escalas dispares) en un formato matemático óptimo para alimentar los modelos de Machine Learning.

**Pasos clave:**
1.  **Selección de características:** separar metadatos informativos de las variables predictivas.
2.  **Codificación (Encoding):** convertir variables categóricas en numéricas.
3.  **Escalado (Scaling):** normalizar rangos numéricos para evitar sesgos por magnitud.

In [1]:
# 1. Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de preprocesamiento de scikit-learn
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# 2. Carga del dataset
df = pd.read_csv('data/SpotifyFeatures_cleaned.csv')

# Verficamos que todo se ha cargado correctamente
print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

Dataset cargado con 174582 filas y 18 columnas.


,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.611,0.389,99373,0.910,0.000,C#,0.3460,-1.828,Major,0.0525,166.969,4/4,0.814
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,0.246,0.590,137373,0.737,0.000,F#,0.1510,-5.559,Minor,0.0868,174.003,4/4,0.816
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,0.952,0.663,170267,0.131,0.000,C,0.1030,-13.879,Minor,0.0362,99.488,5/4,0.368
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.703,0.240,152427,0.326,0.000,C#,0.0985,-12.178,Major,0.0395,171.758,4/4,0.227
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,0.950,0.331,82625,0.225,0.123,F,0.2020,-21.150,Major,0.0456,140.576,4/4,0.390


### **2.1 Selección de características (Features Selection)**

Basado en las conclusiones del EDA, procedemos a:
1. Eliminar `time_signature` por su baja varianza.
2. Separar los metadatos (`artist_name`, `track_name`, `popularity`) de las características numéricas de audio. Necesitamos las etiquetas para mostrar resultados, pero el modelo matemático solo "verá" los números.

In [2]:
# 1. Eliminamos columnas irrelevantes para el modelo
df_model = df.drop(columns=['time_signature'])

# 2. Seleccionamos las columnas de metadatos (lo que usaremos para mostrar resultados en la app)
metadata_cols = ['track_id', 'artist_name', 'track_name', 'popularity']
df_metadata = df_model[metadata_cols]

# 3. Seleccionamos las columnas de características (features) para el modelo
df_features = df_model.drop(columns=metadata_cols)

print("\n--- División de Datos ---")
print(f"Metadatos guardados (Filas, Cols): {df_metadata.shape}")
print(f"Features para el modelo (Filas, Cols): {df_features.shape}")
print("\nColumnas para el modelo:")
print(df_features.columns.tolist())


--- División de Datos ---
Metadatos guardados (Filas, Cols): (174582, 4)
Features para el modelo (Filas, Cols): (174582, 13)

Columnas para el modelo:
['genre', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence']


Si observamos bien las columnas seleccionadas para el modelo, podemos ver que aún hay 2 variables categóricas (`key`, `mode`), que deberán ser transformadas.

In [3]:
df_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174582 entries, 0 to 174581
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   genre             174582 non-null  object 
 1   acousticness      174582 non-null  float64
 2   danceability      174582 non-null  float64
 3   duration_ms       174582 non-null  int64  
 4   energy            174582 non-null  float64
 5   instrumentalness  174582 non-null  float64
 6   key               174582 non-null  object 
 7   liveness          174582 non-null  float64
 8   loudness          174582 non-null  float64
 9   mode              174582 non-null  object 
 10  speechiness       174582 non-null  float64
 11  tempo             174582 non-null  float64
 12  valence           174582 non-null  float64
dtypes: float64(9), int64(1), object(3)
memory usage: 17.3+ MB


### **2.2 Codificación de variables categóricas (Encoding)**

Las variables `genre`, `key` y `mode` son categóricas. Para que los algoritmos de distancia (como euclídea o coseno) funcionen, aplicamos **One-Hot Encoding**.
* Esto creará una columna binaria por cada tonalidad (ej: `key_C`, `key_F#`).
* Evita introducir un falso orden ordinal (no queremos que el modelo piense que la nota Si es "mayor" que Do).

In [4]:
# Separamos las columnas categóricas de las numéricas en df_features
categorical_cols = ['genre','key', 'mode']
numerical_cols = ['acousticness', 'danceability', 'duration_ms', 
                  'energy', 'instrumentalness', 'liveness', 'loudness', 
                  'speechiness', 'tempo', 'valence']

# Aplicamos One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Entrenamos y transformamos
encoded_data = encoder.fit_transform(df_features[categorical_cols])

# Creamos un DataFrame con las nuevas columnas
encoded_cols = encoder.get_feature_names_out(categorical_cols)
df_encoded = pd.DataFrame(encoded_data, columns=encoded_cols)

# Reseteamos índices para poder concatenar sin errores
df_features.reset_index(drop=True, inplace=True)
df_encoded.reset_index(drop=True, inplace=True)

print("Codificación completada. Nuevas columnas generadas:")
print(encoded_cols)

Codificación completada. Nuevas columnas generadas:
['genre_A Capella' 'genre_Alternative' 'genre_Anime' 'genre_Blues'
 "genre_Children's Music" 'genre_Children’s Music' 'genre_Classical'
 'genre_Comedy' 'genre_Country' 'genre_Dance' 'genre_Electronic'
 'genre_Folk' 'genre_Hip-Hop' 'genre_Indie' 'genre_Jazz' 'genre_Movie'
 'genre_Opera' 'genre_Pop' 'genre_R&B' 'genre_Rap' 'genre_Reggae'
 'genre_Reggaeton' 'genre_Rock' 'genre_Ska' 'genre_Soul'
 'genre_Soundtrack' 'genre_World' 'key_A' 'key_A#' 'key_B' 'key_C'
 'key_C#' 'key_D' 'key_D#' 'key_E' 'key_F' 'key_F#' 'key_G' 'key_G#'
 'mode_Major' 'mode_Minor']


### **2.3 Escalado de características numéricas (Scaling)**

Para que el modelo de distancia (KNN/Coseno) trate a todas las variables con la misma importancia, debemos llevarlas a una escala común.
* Variables como `duration_ms` (rango 0 - 5000000) dominarían sobre variables pequeñas como `energy` (rango 0 - 1).
* **Solución:** Aplicamos **MinMaxScaler** para transformar todas las variables numéricas al rango exacto **[0, 1]**.

In [5]:
# Inicializamos el Scaler
scaler = MinMaxScaler()

# Ajustamos y transformamos las columnas numéricas
scaled_data = scaler.fit_transform(df_features[numerical_cols])

# Convertimos a DataFrame manteniendo los nombres de columnas
df_scaled = pd.DataFrame(scaled_data, columns=numerical_cols)

print("Escalado completado. Muestra de los datos:")
df_scaled.head()

Escalado completado. Muestra de los datos:


,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
0,0.613454,0.356223,0.121687,0.910909,0.000000,0.339614,0.900856,0.032070,0.642704,0.814
1,0.246988,0.571888,0.188355,0.737732,0.000000,0.142710,0.834469,0.068374,0.675801,0.816
2,0.955823,0.650215,0.246065,0.131113,0.000000,0.094241,0.686429,0.014818,0.325182,0.368
3,0.705823,0.196352,0.214766,0.326313,0.000000,0.089697,0.716695,0.018311,0.665238,0.227
4,0.953815,0.293991,0.092304,0.225209,0.123123,0.194208,0.557054,0.024767,0.518516,0.390


Hecho esto, ahora tenemos 3 piezas sueltas:
1. `df_metadata` (Nombres, IDs, géneros)
2. `df_scaled` (Variables numéricas escaladas)
3. `df_encoded` (Variables categóricas convertidas a One-Hot)

Estas las debemos juntar en un dataset procesado final.

### **2.4 Unificación y exportación de modelos**

Concatenamos todas las piezas (metadatos + numéricas Escaladas + categóricas codificadas) para generar el dataset final que consumirá el modelo.

Adicionalmente, **exportamos los objetos `scaler` y `encoder`**.
* **Motivo:** La app necesitará estos objetos exactos para procesar las nuevas canciones que introduzca el usuario o vengan de la API, aplicando las mismas transformaciones matemáticas que usamos en el entrenamiento.

In [6]:
import joblib # Librería estándar para guardar modelos de IA
import os

# Creación de estructura de carpetas si no existen
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# 1. Concatenar todo en un solo DataFrame final
df_final_processed = pd.concat([df_metadata, df_scaled, df_encoded], axis=1)

# 2. Guardar el dataset procesado
df_final_processed.to_csv("data/dataset_procesado.csv", index=False)
print("Archivo guardado en: data/dataset_procesado.csv")

# 3. Guardar los objetos de preprocesamiento (Pipeline)
# Esto es CRUCIAL para la App
joblib.dump(scaler, 'models/modelo_scaler.joblib')
joblib.dump(encoder, 'models/modelo_encoder.joblib')
joblib.dump(numerical_cols, 'models/config_num_cols.joblib') # Guardamos los nombres de columnas para no equivocarnos luego
joblib.dump(categorical_cols, 'models/config_cat_cols.joblib')

print("Modelos guardados en la carpeta 'models/'.")

Archivo guardado en: data/dataset_procesado.csv
Modelos guardados en la carpeta 'models/'.
